In [1]:
import pandas as pd
import numpy as np
import joblib

FEAT_PATH = r"C:\Users\santi\Documents\pyrolisis\Data\pyrolysis_features.csv"
MODEL_PATH = r"C:\Users\santi\Documents\pyrolisis\models\xgb_liquid.pkl"

df = pd.read_csv(FEAT_PATH)
model = joblib.load(MODEL_PATH)

print("Dataset:", df.shape)
print("Modelo cargado:", type(model))

Dataset: (512, 21)
Modelo cargado: <class 'xgboost.sklearn.XGBRegressor'>


|model.predict(X) - target|
restricciones físicas para que el optimizador no invente biomasas imposibles:

VM + FC + Ash + M ≈ 100 (cierre del análisis próximo)
C + H + O + N ≈ 100 (cierre elemental, pero H/C y O/C son los que entran al modelo)
Todos los valores positivos
FT entre 400 y 700 (rango razonable)


In [2]:
from scipy.optimize import minimize

features = ['H_C', 'O_C', 'N', 'Ash ', 'VM', 'FC', 'M', 
            'FT', 'FT_squared', 'HR', 'PS', 'FR', 'FR_disponible']

# Punto inicial: mediana de cada feature
x0 = df[features].median().values

print("Punto inicial:")
for f, v in zip(features, x0):
    print(f"  {f}: {v:.3f}")

Punto inicial:
  H_C: 1.648
  O_C: 0.670
  N: 1.810
  Ash : 5.500
  VM: 74.800
  FC: 14.300
  M: 5.600
  FT: 500.000
  FT_squared: 250000.000
  HR: 15.000
  PS: 0.750
  FR: 100.000
  FR_disponible: 1.000


In [3]:
TARGET = 50.0  # % de bio-oil deseado

def objetivo(x):
    X = pd.DataFrame([x], columns=features)
    pred = model.predict(X)[0]
    return (pred - TARGET) ** 2

# Restricciones físicas
constraints = [
    # VM + FC + Ash + M ≈ 100
    {'type': 'eq', 'fun': lambda x: x[features.index('VM')] + 
                                     x[features.index('FC')] + 
                                     x[features.index('Ash ')] + 
                                     x[features.index('M')] - 100},
    # FT_squared debe ser consistente con FT
    {'type': 'eq', 'fun': lambda x: x[features.index('FT_squared')] - 
                                     x[features.index('FT')] ** 2}
]

# Bounds: límites físicos por variable
bounds = [
    (0.5, 4.0),    # H_C
    (0.1, 1.5),    # O_C
    (0.0, 10.0),   # N
    (0.0, 40.0),   # Ash
    (10.0, 90.0),  # VM
    (1.0, 60.0),   # FC
    (0.0, 20.0),   # M
    (300.0, 800.0),# FT
    (None, None),  # FT_squared — controlado por constraint
    (1.0, 100.0),  # HR
    (0.1, 5.0),    # PS
    (10.0, 500.0), # FR
    (1.0, 1.0),    # FR_disponible — fijo en 1 (medición real)
]

In [4]:
result = minimize(
    objetivo,
    x0,
    method='SLSQP',
    bounds=bounds,
    constraints=constraints,
    options={'maxiter': 1000, 'ftol': 1e-6}
)

print("Optimización exitosa:", result.success)
print("Mensaje:", result.message)
print(f"Predicción con solución óptima: {model.predict(pd.DataFrame([result.x], columns=features))[0]:.2f}%")
print(f"Target: {TARGET}%")

Optimización exitosa: True
Mensaje: Optimization terminated successfully
Predicción con solución óptima: 49.20%
Target: 50.0%


In [5]:
solucion = pd.Series(result.x, index=features)

print("=== BIOMASA ÓPTIMA PARA ~50% BIO-OIL ===\n")
print("Composición elemental:")
print(f"  H/C:  {solucion['H_C']:.3f}")
print(f"  O/C:  {solucion['O_C']:.3f}")
print(f"  N:    {solucion['N']:.2f}%")

print("\nAnálisis próximo:")
print(f"  VM:   {solucion['VM']:.1f}%")
print(f"  FC:   {solucion['FC']:.1f}%")
print(f"  Ash:  {solucion['Ash ']:.1f}%")
print(f"  M:    {solucion['M']:.1f}%")

print("\nCondiciones de proceso:")
print(f"  FT:   {solucion['FT']:.0f} °C")
print(f"  HR:   {solucion['HR']:.1f} °C/min")
print(f"  PS:   {solucion['PS']:.2f} mm")
print(f"  FR:   {solucion['FR']:.0f} ml/min")

=== BIOMASA ÓPTIMA PARA ~50% BIO-OIL ===

Composición elemental:
  H/C:  1.648
  O/C:  0.670
  N:    1.81%

Análisis próximo:
  VM:   74.8%
  FC:   14.3%
  Ash:  5.5%
  M:    5.6%

Condiciones de proceso:
  FT:   500 °C
  HR:   15.0 °C/min
  PS:   0.75 mm
  FR:   100 ml/min


In [7]:
# Punto inicial: percentil 25 en lugar de mediana
x0_bajo = df[features].quantile(0.25).values.copy()

# Corregir FT_squared para que sea consistente con FT
idx_ft = features.index('FT')
idx_ft2 = features.index('FT_squared')
x0_bajo[idx_ft2] = x0_bajo[idx_ft] ** 2

result2 = minimize(
    objetivo,
    x0_bajo,
    method='SLSQP',
    bounds=bounds,
    constraints=constraints,
    options={'maxiter': 1000, 'ftol': 1e-6}
)

print("Optimización exitosa:", result2.success)
print(f"Predicción: {model.predict(pd.DataFrame([result2.x], columns=features))[0]:.2f}%")

solucion2 = pd.Series(result2.x, index=features)
print("\n=== BIOMASA ÓPTIMA (desde punto bajo) ===\n")
print(f"  H/C:  {solucion2['H_C']:.3f}")
print(f"  O/C:  {solucion2['O_C']:.3f}")
print(f"  N:    {solucion2['N']:.2f}%")
print(f"  VM:   {solucion2['VM']:.1f}%")
print(f"  FC:   {solucion2['FC']:.1f}%")
print(f"  Ash:  {solucion2['Ash ']:.1f}%")
print(f"  M:    {solucion2['M']:.1f}%")
print(f"  FT:   {solucion2['FT']:.0f} °C")
print(f"  HR:   {solucion2['HR']:.1f} °C/min")

Optimización exitosa: True
Predicción: 42.26%

=== BIOMASA ÓPTIMA (desde punto bajo) ===

  H/C:  1.470
  O/C:  0.577
  N:    0.75%
  VM:   73.5%
  FC:   13.7%
  Ash:  5.5%
  M:    7.3%
  FT:   450 °C
  HR:   10.0 °C/min


In [8]:
TARGET = 60.0

result3 = minimize(
    objetivo,
    x0.copy(),
    method='SLSQP',
    bounds=bounds,
    constraints=constraints,
    options={'maxiter': 1000, 'ftol': 1e-6}
)

print("Optimización exitosa:", result3.success)
print(f"Predicción: {model.predict(pd.DataFrame([result3.x], columns=features))[0]:.2f}%")

solucion3 = pd.Series(result3.x, index=features)
print("\n=== BIOMASA ÓPTIMA PARA ~60% BIO-OIL ===\n")
print(f"  H/C:  {solucion3['H_C']:.3f}")
print(f"  O/C:  {solucion3['O_C']:.3f}")
print(f"  N:    {solucion3['N']:.2f}%")
print(f"  VM:   {solucion3['VM']:.1f}%")
print(f"  FC:   {solucion3['FC']:.1f}%")
print(f"  Ash:  {solucion3['Ash ']:.1f}%")
print(f"  M:    {solucion3['M']:.1f}%")
print(f"  FT:   {solucion3['FT']:.0f} °C")
print(f"  HR:   {solucion3['HR']:.1f} °C/min")

Optimización exitosa: True
Predicción: 49.20%

=== BIOMASA ÓPTIMA PARA ~60% BIO-OIL ===

  H/C:  1.648
  O/C:  0.670
  N:    1.81%
  VM:   74.8%
  FC:   14.3%
  Ash:  5.5%
  M:    5.6%
  FT:   500 °C
  HR:   15.0 °C/min


In [12]:
# Optimizamos sin VM — lo calculamos como 100 - FC - Ash - M
features_opt = ['H_C', 'O_C', 'N', 'Ash ', 'FC', 'M', 
                'FT', 'HR', 'PS', 'FR']

bounds_reducidos = [
    (0.5, 4.0),    # H_C
    (0.1, 1.5),    # O_C
    (0.0, 10.0),   # N
    (0.0, 30.0),   # Ash
    (1.0, 50.0),   # FC
    (0.0, 20.0),   # M
    (300.0, 800.0),# FT
    (1.0, 100.0),  # HR
    (0.1, 5.0),    # PS
    (10.0, 500.0), # FR
]

def objetivo_reducido(x):
    h_c, o_c, n, ash, fc, m, ft, hr, ps, fr = x
    vm = 100 - fc - ash - m
    if vm < 5 or vm > 90:  # VM fuera de rango físico
        return 1e6
    ft_sq = ft ** 2
    X = pd.DataFrame([[h_c, o_c, n, ash, vm, fc, m, ft, ft_sq, hr, ps, fr, 1.0]], 
                     columns=features)
    pred = model.predict(X)[0]
    return (pred - 60.0) ** 2

result5 = differential_evolution(
    objetivo_reducido, bounds_reducidos, 
    seed=42, maxiter=300, tol=1e-4, 
    disp=True, popsize=10
)

print(f"\nConvergió: {result5.success}")
h_c,o_c,n,ash,fc,m,ft,hr,ps,fr = result5.x
vm = 100 - fc - ash - m
ft_sq = ft**2
X_opt = pd.DataFrame([[h_c,o_c,n,ash,vm,fc,m,ft,ft_sq,hr,ps,fr,1.0]], columns=features)
pred = model.predict(X_opt)[0]
print(f"Predicción: {pred:.2f}%")
print(f"\n  H/C: {h_c:.3f} | O/C: {o_c:.3f} | N: {n:.2f}%")
print(f"  VM:  {vm:.1f}% | FC: {fc:.1f}% | Ash: {ash:.1f}% | M: {m:.1f}%")
print(f"  FT:  {ft:.0f}°C | HR: {hr:.1f}°C/min")

differential_evolution step 1: f(x)= 118.19818115234375
differential_evolution step 2: f(x)= 84.30911254882812
differential_evolution step 3: f(x)= 33.56361389160156
differential_evolution step 4: f(x)= 33.01811981201172
differential_evolution step 5: f(x)= 4.369344234466553
differential_evolution step 6: f(x)= 4.369344234466553
differential_evolution step 7: f(x)= 2.4206011295318604
differential_evolution step 8: f(x)= 0.9839516282081604
differential_evolution step 9: f(x)= 0.9839516282081604
differential_evolution step 10: f(x)= 0.9839516282081604
differential_evolution step 11: f(x)= 0.9839516282081604
differential_evolution step 12: f(x)= 0.9839516282081604
differential_evolution step 13: f(x)= 0.9839516282081604
differential_evolution step 14: f(x)= 0.0018273484893143177
differential_evolution step 15: f(x)= 0.0018273484893143177
differential_evolution step 16: f(x)= 0.0018273484893143177
differential_evolution step 17: f(x)= 0.0018273484893143177
differential_evolution step 18: f

Biomasa “óptima” — interpretación y alcance

El modelo converge hacia una biomasa con H/C = 1.170, lo que químicamente implica mayor insaturación y carácter aromático, condición que en principio favorecería formación de char más que de bio-oil. Sin embargo, esa desventaja estructural se ve compensada por un VM = 78.1%, que incrementa la fracción susceptible de despolimerización primaria y generación de vapores condensables. La solución refleja una tensión fisicoquímica real: la composición elemental no es ideal para líquido, pero la alta fracción volátil domina el balance global de fases.

La temperatura óptima estimada (557 °C) se ubica muy cerca del máximo teórico previamente calculado (568 °C) mediante un ajuste parabólico independiente. La convergencia de ambos enfoques constituye una validación cruzada robusta del rango térmico óptimo para maximizar líquidos.

Es importante enfatizar que esta biomasa es una solución matemática dentro del espacio del modelo; no implica que exista en la naturaleza exactamente con esa combinación composicional. Debe interpretarse como una guía para diseño experimental y screening dirigido, no como una receta directa de formulación.

In [13]:
solucion_final = pd.DataFrame({
    'feature': features,
    'valor_optimo': X_opt.values[0]
})
solucion_final.to_csv(r"C:\Users\santi\Documents\pyrolisis\Data\biomasa_optima.csv", index=False)
print("Guardado")

Guardado
